In [2]:
import pandas as pd
import numpy as np
from pyomo.environ import *

file_name = "C:/Users/Enterprise-CV6/Desktop/AAMA/2026 World Cup Data.xlsx"
teams = pd.read_excel(file_name, sheet_name="Teams")

teams.columns = [c.strip() for c in teams.columns]
need = ["Nation", "Code", "Confederation", "FIFA points", "Pot"]

teams = teams.dropna(subset=need).copy()
teams["FIFA points"] = pd.to_numeric(teams["FIFA points"], errors="coerce")
teams["Pot"] = pd.to_numeric(teams["Pot"], errors="coerce")
teams["Confederation"] = teams["Confederation"].astype(str).str.strip().str.upper()

teams = teams.dropna(subset=["FIFA points","Pot"]).copy()
teams = teams.reset_index(drop=True)
teams["TeamID"] = teams.index

T_set = teams["TeamID"].tolist()
G_set = list(range(12))  # 12 groups

points = dict(zip(teams["TeamID"], teams["FIFA points"]))
pot = dict(zip(teams["TeamID"], teams["Pot"].astype(int)))
conf = dict(zip(teams["TeamID"], teams["Confederation"].astype(str).str.strip().str.upper()))

confeds = sorted(teams["Confederation"].astype(str).str.strip().str.upper().unique().tolist())
avg_points = teams["FIFA points"].sum() / 12.0

model = ConcreteModel()

# sets
model.T = Set(initialize=T_set)
model.G = Set(initialize=G_set)
model.P = Set(initialize=[1,2,3,4])
model.Cnon = Set(initialize=[c for c in confeds if c != "UEFA"])

# ========= Decision Variables =========
# x[t,g] = 1 if team t assigned to group g
model.x = Var(model.T, model.G, domain=Binary)

# group points sum
model.S = Var(model.G, domain=NonNegativeReals)

# minimum group total points
model.w = Var(domain=NonNegativeReals)

# ========= Objective =========
# maximize the minimum group total points
def obj_rule(m):
    return m.w
model.objective = Objective(rule=obj_rule, sense=maximize)

# ========= Constraints =========

# (a) each team in one group
def one_group_rule(m, t):
    return sum(m.x[t,g] for g in m.G) == 1
model.one_group = Constraint(model.T, rule=one_group_rule)

# (b) each group has four teams
def group_size_rule(m, g):
    return sum(m.x[t,g] for t in m.T) == 4
model.group_size = Constraint(model.G, rule=group_size_rule)

# (c) each group have one pot X(1,2,3,4)
def pot_rule(m, g, p):
    return sum(m.x[t,g] for t in m.T if pot[t] == p) == 1
model.pot_cons = Constraint(model.G, model.P, rule=pot_rule)

# (d) Confederation: UEFA <= 2
def uefa_rule(m, g):
    return sum(m.x[t,g] for t in m.T if conf[t] == "UEFA") <= 2
model.uefa_cons = Constraint(model.G, rule=uefa_rule)

# (e) Confederation: not UEFA <= 1
def nonuefa_rule(m, g, c):
    return sum(m.x[t,g] for t in m.T if conf[t] == c) <= 1
model.nonuefa_cons = Constraint(model.G, model.Cnon, rule=nonuefa_rule)

# (f) each group points in total(S[G])
def sum_points_rule(m, g):
    return m.S[g] == sum(points[t]*m.x[t,g] for t in m.T)
model.sum_points = Constraint(model.G, rule=sum_points_rule)

# (g) Chebyshev linking constraints: w <= S[g] for all groups g
def chebyshev_rule(m, g):
    return m.w <= m.S[g]
model.chebyshev = Constraint(model.G, rule=chebyshev_rule)

# Symmetry-breaking constraints

# The 12 groups (A–L) are structurally identical, which creates
# a large number of symmetric (equivalent) solutions.
#
# To reduce the search space and improve computational speed,
# we fix the Pot 1 teams (seeded teams) to specific groups.
# Each group must already contain exactly one Pot 1 team,
# so fixing them does not affect feasibility or fairness.
#
# We assign Pot 1 teams to groups in descending FIFA points order:
#   strongest Pot 1 team  -> Group 0 (A)
#   second strongest      -> Group 1 (B)
#   ...
#   12th strongest        -> Group 11 (L)
# ============================================================

# Extract Pot 1 teams
pot1_teams = [t for t in T_set if pot[t] == 1]

# Sort Pot 1 teams by FIFA points (descending)
pot1_sorted = sorted(pot1_teams, key=lambda t: -points[t])

# Fix assignment: i-th strongest Pot 1 team -> group i
for i, t in enumerate(pot1_sorted):
    model.x[t, i].fix(1)

solver = SolverFactory("glpk")
results = solver.solve(model, tee=True)   # tee=True, print answer

tc = results.solver.termination_condition
print("Termination:", tc)

if tc != TerminationCondition.optimal:
    raise RuntimeError(f"no optimal，termination={tc}")

print("Optimal w (minimum group points) =", value(model.w))

# Extract the members of each anonymous group (g = 0..11)
groups = []
for g in G_set:
    members = []
    for t in T_set:
        if value(model.x[t, g]) > 0.5:
            r = teams.loc[teams["TeamID"]==t,
                          ["Nation","Code","Confederation","Pot","FIFA points"]
                         ].iloc[0].to_dict()
            members.append(r)

    # Sort teams by Pot number (Pot 1–4 correspond to Seed1–Seed4)
    members = sorted(members, key=lambda x: int(x["Pot"]))

    # Compute total FIFA points of the group
    group_points = sum(float(m["FIFA points"]) for m in members)

    # Store group result in a single row (anonymous group ID 1–12)
    row = {
        "GroupID": g+1,  # Anonymous group index (not letter assignment yet)
        "Seed1 (Pot1)": members[0]["Nation"],
        "Seed2 (Pot2)": members[1]["Nation"],
        "Seed3 (Pot3)": members[2]["Nation"],
        "Seed4 (Pot4)": members[3]["Nation"],
        "GroupPoints": group_points,
    }
    groups.append(row)

# Create DataFrame of grouping results
group_df = pd.DataFrame(groups)

# Optional: sort groups by total points (useful for balance comparison)
group_df = group_df.sort_values("GroupPoints", ascending=False).reset_index(drop=True)

# Reassign anonymous GroupID after sorting
group_df["GroupID"] = range(1, len(group_df)+1)

# Display results
print(group_df.to_string(index=False))
min_gp = group_df["GroupPoints"].min()
max_gp = group_df["GroupPoints"].max()
avg_gp = group_df["GroupPoints"].mean()
max_dev = (group_df["GroupPoints"] - avg_gp).abs().max()

print("\nOptimal w (minimum group points) =", float(value(model.w)))
print("Min(GroupPoints) =", float(min_gp))
print("Max(GroupPoints) =", float(max_gp))
print("Range (max-min) =", float(max_gp - min_gp))
print("Standard deviation of group totals =", float(group_df["GroupPoints"].std()))
print("Max deviation from average (post-computed) =", float(max_dev))


# =========================
# Comparison: FIFA proposed formation (A–L)
# =========================

# Build Nation -> FIFA points lookup
teams["Nation"] = teams["Nation"].astype(str).str.strip()
points_by_nation = dict(zip(teams["Nation"], teams["FIFA points"]))

# Input the current FIFA proposed formation (A–L)

fifa_groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Denmark"],
    "B": ["Canada", "Italy", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"], 
    "F": ["Netherlands", "Japan", "Ukraine", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

# Compute baseline group totals
baseline_rows = []
for grp, members in fifa_groups.items():
    total = 0.0
    for name in members:
        try:
            total += float(points_by_nation[name])
        except KeyError:
            raise KeyError(f"Name '{name}' in Group {grp} not found in Teams['Nation']. "
                           f"Please change it to match the exact spelling in your Excel.")
    baseline_rows.append({"Group": grp, "GroupPoints": total})

baseline_df = pd.DataFrame(baseline_rows).sort_values("Group").reset_index(drop=True)
print("\n=== FIFA Proposed Formation: Group Points ===")
print(baseline_df.to_string(index=False))

# Baseline metrics
w_base = baseline_df["GroupPoints"].min()
range_base = baseline_df["GroupPoints"].max() - baseline_df["GroupPoints"].min()
std_base = baseline_df["GroupPoints"].std()
avg_base = baseline_df["GroupPoints"].mean()
max_dev_base = (baseline_df["GroupPoints"] - avg_base).abs().max()

print("\n=== Baseline (FIFA Proposed) Metrics ===")
print("w_base (min group points) =", round(float(w_base), 2))
print("range_base (max-min)      =", round(float(range_base), 2))
print("std_base                  =", round(float(std_base), 2))
print("max_dev_from_avg_base     =", round(float(max_dev_base), 2))

# Optimized metrics (from your group_df)
w_opt = group_df["GroupPoints"].min()
range_opt = group_df["GroupPoints"].max() - group_df["GroupPoints"].min()
std_opt = group_df["GroupPoints"].std()
avg_opt = group_df["GroupPoints"].mean()
max_dev_opt = (group_df["GroupPoints"] - avg_opt).abs().max()

print("\n=== Optimized (Our Model) Metrics ===")
print("w_opt (min group points) =", round(float(w_opt), 2))
print("range_opt (max-min)      =", round(float(range_opt), 2))
print("std_opt                  =", round(float(std_opt), 2))
print("max_dev_from_avg_opt     =", round(float(max_dev_opt), 2))

print("\n=== Improvement (Optimized - Baseline) ===")
print("Δw (min group points)    =", round(float(w_opt - w_base), 2))
print("Δrange                   =", round(float(range_opt - range_base), 2))
print("Δstd                     =", round(float(std_opt - std_base), 2))
print("Δmax_dev_from_avg        =", round(float(max_dev_opt - max_dev_base), 2))

GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 --write C:\Users\ENTERP~1\AppData\Local\Temp\tmpgpi2exfi.glpk.raw --wglp
 C:\Users\ENTERP~1\AppData\Local\Temp\tmppilmh61k.glpk.glp --cpxlp C:\Users\ENTERP~1\AppData\Local\Temp\tmprtihjxhx.pyomo.lp
Reading problem data from 'C:\Users\ENTERP~1\AppData\Local\Temp\tmprtihjxhx.pyomo.lp'...
C:\Users\ENTERP~1\AppData\Local\Temp\tmprtihjxhx.pyomo.lp:4056: warning: lower bound of variable 'x4' redefined
C:\Users\ENTERP~1\AppData\Local\Temp\tmprtihjxhx.pyomo.lp:4056: warning: upper bound of variable 'x4' redefined
204 rows, 577 columns, 2856 non-zeros
564 integer variables, all of which are binary
4620 lines were read
Writing problem data to 'C:\Users\ENTERP~1\AppData\Local\Temp\tmppilmh61k.glpk.glp'...
3834 lines were written
GLPK Integer Optimizer, v4.65
204 rows, 577 columns, 2856 non-zeros
564 integer variables, all of which are binary
Preprocessing...
162 rows, 419 columns, 2054 non-zeros
406 integer variables, 

In [3]:
# =========================================================
# PART 3 (Step 1): Compute match popularity inside each numeric group
# Output a table for sanity-checking (NO optimization yet)
# =========================================================
# ---- 0) Clean & build lookups ----
teams["Nation"] = teams["Nation"].astype(str).str.strip()
teams["Code"] = teams["Code"].astype(str).str.strip()

name_to_code = dict(zip(teams["Nation"], teams["Code"]))
f_by_code = dict(zip(teams["Code"], teams["Spectator index"] / 100.0))  # scale to [0,1]

def match_pop(f1, f2):
    """Popularity p in [0,4] per Ghoniem-style definition used in your assignment."""
    other = 0.5 * (f1 + f2)
    officials = 1.0 if (abs(f1 - 1.0) < 1e-12 or abs(f2 - 1.0) < 1e-12) else 0.5 * (f1 + f2)
    return f1 + f2 + other + officials

# ---- 1) Column mapping (edit here if your Part2 output column names differ) ----
seed_cols = {
    1: "Seed1 (Pot1)",
    2: "Seed2 (Pot2)",
    3: "Seed3 (Pot3)",
    4: "Seed4 (Pot4)",
}

required_cols = ["GroupID"] + list(seed_cols.values())
missing = [c for c in required_cols if c not in group_df.columns]
if missing:
    raise KeyError(f"group_df is missing columns: {missing}\n"
                   f"Please edit seed_cols mapping to match your actual column names.")

# ---- 2) Compute 6 match popularities for each group ----
pairs = [(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)]
pop_cols = [f"pop_{a}{b}" for a,b in pairs]

rows_out = []
unknown_nations = set()

for _, row in group_df.iterrows():
    g = int(row["GroupID"])

    # nation names -> codes -> f
    pot_code = {}
    pot_f = {}
    for pot in [1,2,3,4]:
        nation = str(row[seed_cols[pot]]).strip()
        if nation not in name_to_code:
            unknown_nations.add(nation)
            pot_code[pot] = None
            pot_f[pot] = None
        else:
            code = name_to_code[nation]
            pot_code[pot] = code
            if code not in f_by_code:
                raise KeyError(f"Team code '{code}' not found in spectator index lookup.")
            pot_f[pot] = float(f_by_code[code])

    # compute match pops (if any missing, set NaN so you notice)
    pops = {}
    for (a,b) in pairs:
        if pot_f[a] is None or pot_f[b] is None:
            pops[(a,b)] = float("nan")
        else:
            pops[(a,b)] = match_pop(pot_f[a], pot_f[b])

    # group total popularity across all 6 matches (optional but useful check)
    group_total_6 = sum(pops[(a,b)] for (a,b) in pairs if pd.notna(pops[(a,b)]))

    out = {"GroupID": g}
    for (a,b) in pairs:
        out[f"pop_{a}{b}"] = pops[(a,b)]
    out["pop_total_6matches"] = group_total_6

    # also add the team codes + f values for quick sanity-check (optional)
    for pot in [1,2,3,4]:
        out[f"Pot{pot}_Code"] = pot_code[pot]
        out[f"Pot{pot}_f"] = pot_f[pot]

    rows_out.append(out)

pop_df = pd.DataFrame(rows_out).sort_values("GroupID")

# ---- 3) Merge back to group_df for a single inspection table ----
check_df = group_df.merge(pop_df, on="GroupID", how="left")

# Round for readability
round_cols = pop_cols + ["pop_total_6matches"] + [f"Pot{p}_f" for p in [1,2,3,4]]
for c in round_cols:
    if c in check_df.columns:
        check_df[c] = check_df[c].astype(float).round(4)

# ---- 4) Print / display ----
if unknown_nations:
    print("\n[WARNING] These nation names were not found in Teams['Nation'] and produced NaN popularities:")
    for n in sorted(unknown_nations):
        print(" -", n)
    print("Fix spelling / naming so it matches Teams sheet exactly.\n")

# =========================================================
# Compact output: GroupID, 4 teams, total popularity (6 matches)
# =========================================================

summary_df = group_df.copy()
summary_df = summary_df.merge(
    pop_df[["GroupID", "pop_total_6matches"]],
    on="GroupID",
    how="left"
)



summary_df = summary_df[[
    "GroupID",
    "Seed1 (Pot1)",
    "Seed2 (Pot2)",
    "Seed3 (Pot3)",
    "Seed4 (Pot4)",
    "pop_total_6matches"
]].sort_values("GroupID")

print("\n=== Group total popularity (6 matches per group) ===")
print(summary_df.to_string(index=False))


=== Group total popularity (6 matches per group) ===
 GroupID  Seed1 (Pot1) Seed2 (Pot2) Seed3 (Pot3) Seed4 (Pot4)  pop_total_6matches
       1 United States     Colombia       Turkey   Uzbekistan           12.861733
       2         Spain  Switzerland        Egypt        Haiti           12.779507
       3     Argentina        Japan      Algeria  New Zealand           11.282900
       4       Belgium      Croatia      Tunisia Saudi Arabia            7.131576
       5       England      Ecuador      Ukraine   Cape Verde           14.324685
       6      Portugal      Senegal    Australia      Curaçao           14.516939
       7   Netherlands  South Korea       Norway South Africa           13.567289
       8        France         Iran     Paraguay        Ghana            4.876999
       9        Brazil      Denmark     Scotland        Qatar           20.746278
      10       Morocco      Uruguay       Panama       Jordan            9.286380
      11        Mexico        Italy  Ivory C

In [4]:
# =========================================================
# PART 3 (Step 2): Compute row-set popularity given a letter assignment
# for validation
# =========================================================

import pandas as pd
import re

# 1) Read row-set sheet (if not already loaded)
rowset_df = pd.read_excel(file_name, sheet_name="Row-set")

# 2) Build a lookup for match popularity inside each numeric group
#    pop_lookup[(g, a, b)] = popularity
pairs = [(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)]

pop_lookup = {}
for _, r in pop_df.iterrows():
    g = int(r["GroupID"])
    for (a,b) in pairs:
        pop_lookup[(g,a,b)] = float(r[f"pop_{a}{b}"])

# 3) Create a "temporary" letter assignment for checking:
#    GroupIDs sorted mapped to A..L
letters = list("ABCDEFGHIJKL")
G_list = sorted(group_df["GroupID"].astype(int).unique().tolist())
if len(G_list) != 12:
    raise ValueError(f"Expected 12 groups, got {len(G_list)} groups: {G_list}")

assign_check = {int(g): letters[i] for i, g in enumerate(G_list)}  # g -> letter
letter_to_group = {L: g for g, L in assign_check.items()}          # letter -> g

print("\n=== Temporary letter assignment (for validation) ===")
print(pd.DataFrame({"GroupID": G_list, "Letter": [assign_check[g] for g in G_list]}).to_string(index=False))

# 4) Parse row-set codes
code_re = re.compile(r"^\s*([A-L])\s*([1-4])([1-4])\s*$")
day_cols = [c for c in rowset_df.columns if str(c).startswith("Day")]

def parse_code(cell):
    """Return (letter, a, b) or None if empty."""
    if not isinstance(cell, str) or not cell.strip():
        return None
    m = code_re.match(cell)
    if not m:
        raise ValueError(f"Unrecognized row-set code: '{cell}'")
    L = m.group(1)
    a, b = int(m.group(2)), int(m.group(3))
    if a == b:
        raise ValueError(f"Invalid code (same pot vs same pot): '{cell}'")
    if a > b:
        a, b = b, a
    return (L, a, b)

# 5) Compute row-set popularity per stadium (row-set ID = stadium ID)
rows_out = []
for _, r in rowset_df.iterrows():

    stadium_id = int(r["ID"])

    total = 0.0
    n_matches = 0

    for d in day_cols:
        parsed = parse_code(r[d])
        if parsed is None:
            continue

        L, a, b = parsed
        g = letter_to_group[L]
        p = pop_lookup[(g, a, b)]

        total += p
        n_matches += 1

    rows_out.append({
        "stadiumID": stadium_id,
        "matches_in_rowset": n_matches,
        "rowset_popularity": total,
    })

rowset_pop_df = pd.DataFrame(rows_out).sort_values("stadiumID")
rowset_pop_df["rowset_popularity"] = rowset_pop_df["rowset_popularity"].round(3)

print("\n=== Row-set popularity (given temporary A..L assignment) ===")
print(rowset_pop_df.to_string(index=False))

print("\nRow-set min =", float(rowset_pop_df["rowset_popularity"].min()))
print("Row-set max =", float(rowset_pop_df["rowset_popularity"].max()))
print("Row-set range =", float(rowset_pop_df["rowset_popularity"].max()
                                - rowset_pop_df["rowset_popularity"].min()))


=== Temporary letter assignment (for validation) ===
 GroupID Letter
       1      A
       2      B
       3      C
       4      D
       5      E
       6      F
       7      G
       8      H
       9      I
      10      J
      11      K
      12      L

=== Row-set popularity (given temporary A..L assignment) ===
 stadiumID  matches_in_rowset  rowset_popularity
         1                  5             10.941
         2                  5             10.857
         3                  3              6.873
         4                  3              7.414
         5                  4              7.444
         6                  5             15.995
         7                  5              9.117
         8                  4              9.577
         9                  5             12.291
        10                  5              5.308
        11                  5              8.505
        12                  5              9.313
        13                  4          

In [5]:
import re

# -------------------------
# Sets
# -------------------------
G_list = sorted(group_df["GroupID"].astype(int).unique().tolist())  # 12 groups
L_list = list("ABCDEFGHIJKL")                                       # 12 letters
R_list = sorted(rowset_df["ID"].astype(int).unique().tolist())      # 16 stadiums/row-sets

pairs = [(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)]
day_cols = [c for c in rowset_df.columns if str(c).startswith("Day")]

if len(G_list) != 12:
    raise ValueError(f"Expected 12 groups, got {len(G_list)}: {G_list}")

# -------------------------
# Popularity lookup: pop[(g,a,b)]
# -------------------------
pop = {}
for _, r in pop_df.iterrows():
    g = int(r["GroupID"])
    for (a,b) in pairs:
        pop[(g,a,b)] = float(r[f"pop_{a}{b}"])

# -------------------------
# Parse row-set template: pairs_in_rowset[r][L] = list of (a,b)
# -------------------------
code_re = re.compile(r"^\s*([A-L])\s*([1-4])([1-4])\s*$")
pairs_in_rowset = {r: {L: [] for L in L_list} for r in R_list}

for _, rr in rowset_df.iterrows():
    rid = int(rr["ID"])
    for d in day_cols:
        val = rr[d]
        if isinstance(val, str) and val.strip():
            m = code_re.match(val)
            if not m:
                raise ValueError(f"Unrecognized row-set code '{val}' at ID={rid}, col={d}")
            L = m.group(1)
            a, b = int(m.group(2)), int(m.group(3))
            if a == b:
                raise ValueError(f"Invalid code '{val}' (same pot vs same pot) at ID={rid}, col={d}")
            if a > b:
                a, b = b, a
            pairs_in_rowset[rid][L].append((a,b))

# -------------------------
# Use Total column for per-rowset tight Big-M
# y_r <= 4 * Total[r]
# -------------------------
if "Total" not in rowset_df.columns:
    raise KeyError("Row-set sheet must contain a 'Total' column (matches per stadium).")

total_dict = dict(zip(rowset_df["ID"].astype(int), rowset_df["Total"].astype(int)))

# quick sanity: optional
print("Total matches by stadium (from data):", total_dict)

# =========================================================
# Build model (Ghoniem-style)
# =========================================================
model = ConcreteModel()

model.G = Set(initialize=G_list)
model.L = Set(initialize=L_list)
model.R = Set(initialize=R_list)

model.z = Var(model.G, model.L, domain=Binary)          # group -> letter assignment
model.y = Var(model.R, domain=NonNegativeReals)         # row-set popularity
model.wmin = Var(domain=NonNegativeReals)
model.wmax = Var(domain=NonNegativeReals)
model.v = Var(model.R, domain=Binary)                   # selector for maximum row-set

model.Total = Param(model.R, initialize=total_dict, within=NonNegativeIntegers)

# each group assigned to exactly one letter
def one_letter_per_group(m, g):
    return sum(m.z[g, L] for L in m.L) == 1
model.c1 = Constraint(model.G, rule=one_letter_per_group)

# each letter assigned exactly one group
def one_group_per_letter(m, L):
    return sum(m.z[g, L] for g in m.G) == 1
model.c2 = Constraint(model.L, rule=one_group_per_letter)

# define y[r]
def y_def(m, r):
    expr = 0
    for g in m.G:
        for L in m.L:
            plist = pairs_in_rowset[int(r)][str(L)]
            if plist:
                expr += m.z[g, L] * sum(pop[(int(g), a, b)] for (a, b) in plist)
    return m.y[r] == expr
model.c3 = Constraint(model.R, rule=y_def)

# wmin <= y[r]
def wmin_le_y(m, r):
    return m.wmin <= m.y[r]
model.c4 = Constraint(model.R, rule=wmin_le_y)

# wmax >= y[r]
def wmax_ge_y(m, r):
    return m.wmax >= m.y[r]
model.c5 = Constraint(model.R, rule=wmax_ge_y)

# selector: wmax <= y[r] + (4*Total[r])*(1 - v[r])  (tight big-M per rowset)
def wmax_selector(m, r):
    return m.wmax <= m.y[r] + (4 * m.Total[r]) * (1 - m.v[r])
model.c6 = Constraint(model.R, rule=wmax_selector)

model.c7 = Constraint(expr=sum(model.v[r] for r in model.R) == 1)

# objective (per exemplar): maximize wmin + wmax
model.obj = Objective(expr=model.wmin + model.wmax, sense=maximize)

# =========================================================
# Solve
# =========================================================
solver = SolverFactory("highs")
results = solver.solve(model, tee=True)

print("Termination:", results.solver.termination_condition)
print("\n=== Objective values ===")
print("wmin =", value(model.wmin))
print("wmax =", value(model.wmax))
print("wmin+wmax =", value(model.obj))

# =========================================================
# Extract optimal assignment
# =========================================================
assign = {}
for g in model.G:
    for L in model.L:
        if value(model.z[g, L]) > 0.5:
            assign[int(g)] = str(L)

assign_df = group_df.copy()
assign_df["Letter"] = assign_df["GroupID"].map(assign)
assign_df = assign_df.sort_values("Letter")

print("\n=== Optimal letter assignment (Group -> A..L) ===")
print(assign_df[["Letter","GroupID","Seed1 (Pot1)","Seed2 (Pot2)","Seed3 (Pot3)","Seed4 (Pot4)"]].to_string(index=False))

# =========================================================
# Stadium popularity under optimal assignment
# =========================================================
stadium_df = pd.DataFrame(
    [{"stadiumID": int(r), "matches": int(value(model.Total[r])), "rowset_popularity": value(model.y[r])} for r in model.R]
).sort_values("stadiumID")
stadium_df["rowset_popularity"] = stadium_df["rowset_popularity"].round(3)

print("\n=== Stadium popularity under optimal assignment ===")
print(stadium_df.to_string(index=False))
print("\nmin =", float(stadium_df["rowset_popularity"].min()))
print("max =", float(stadium_df["rowset_popularity"].max()))
print("range =", float(stadium_df["rowset_popularity"].max() - stadium_df["rowset_popularity"].min()))

Total matches by stadium (from data): {1: 5, 2: 5, 3: 3, 4: 3, 5: 4, 6: 5, 7: 5, 8: 4, 9: 5, 10: 5, 11: 5, 12: 5, 13: 4, 14: 5, 15: 5, 16: 4}
Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
MIP has 89 rows; 178 cols; 1044 nonzeros; 160 integer variables (160 binary)
Coefficient ranges:
  Matrix  [1e-01, 2e+01]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 2e+01]
Presolving model
89 rows, 178 cols, 1044 nonzeros  0s
89 rows, 178 cols, 987 nonzeros  0s
Presolve reductions: rows 89(-0); columns 178(-0); nonzeros 987(-57) 

Solving MIP model with:
   89 rows
   178 cols (160 binary, 0 integer, 0 implied int., 18 continuous, 0 domain fixed)
   987 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
 

In [11]:
# =========================================================
# PART 4: Assign row-sets to stadiums
# Directly use Part 3 row-set popularity
# =========================================================

# 1) row-set popularity from Part 3
# rowset_pop_df should look like:
# rowsetID, matches, rowset_popularity

R = rowset_pop_df["rowsetID"].astype(int).tolist()

# 2) stadiums from Stadiums sheet
# stadiums_data index = stadium name
S = stadiums_data.index.tolist()

# 3) parameters
# row-set popularity
w = dict(zip(rowset_pop_df["rowsetID"].astype(int),
             rowset_pop_df["rowset_popularity"].astype(float)))

# stadium capacity
c = stadiums_data["Capacity"].to_dict()

# city
city_name = stadiums_data["City"].to_dict()

# 4) build model
model4 = ConcreteModel()

model4.x = Var(R, S, domain=Binary)

# objective: assign high-popularity row-sets to high-capacity stadiums
model4.obj = Objective(
    expr=sum(c[s] * w[r] * model4.x[r, s] for r in R for s in S),
    sense=maximize
)

# each stadium gets exactly one row-set
def rule1(model, s):
    return sum(model.x[r, s] for r in R) == 1
model4.constr1 = Constraint(S, rule=rule1)

# each row-set gets exactly one stadium
def rule2(model, r):
    return sum(model.x[r, s] for s in S) == 1
model4.constr2 = Constraint(R, rule=rule2)

# 5) solve
solver = SolverFactory("glpk")
results = solver.solve(model4, tee=True)

# 6) output
assign_out = []

for r in R:
    for s in S:
        if value(model4.x[r, s]) > 0.5:
            assign_out.append({
                "rowsetID": int(r),
                "stadium": s,
                "city": city_name[s],
                "rowset_popularity": float(w[r]),
                "capacity": float(c[s]),
                "pair_value": float(w[r] * c[s])
            })

assign_df_part4 = pd.DataFrame(assign_out).sort_values("rowsetID")
print(assign_df_part4.to_string(index=False))

print("\nTotal popularity value =", value(model4.obj))

# baseline: assign rowsets to stadiums in current sheet order
baseline_total = sum(c[s] * w[r] for r, s in zip(R, S))
print("Baseline total value =", baseline_total)

GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 --write C:\Users\ENTERP~1\AppData\Local\Temp\tmpervrmzuf.glpk.raw --wglp
 C:\Users\ENTERP~1\AppData\Local\Temp\tmp90foqqr6.glpk.glp --cpxlp C:\Users\ENTERP~1\AppData\Local\Temp\tmp0y3x072w.pyomo.lp
Reading problem data from 'C:\Users\ENTERP~1\AppData\Local\Temp\tmp0y3x072w.pyomo.lp'...
C:\Users\ENTERP~1\AppData\Local\Temp\tmp0y3x072w.pyomo.lp:1130: warning: lower bound of variable 'x2' redefined
C:\Users\ENTERP~1\AppData\Local\Temp\tmp0y3x072w.pyomo.lp:1130: warning: upper bound of variable 'x2' redefined
32 rows, 256 columns, 512 non-zeros
256 integer variables, all of which are binary
1386 lines were read
Writing problem data to 'C:\Users\ENTERP~1\AppData\Local\Temp\tmp90foqqr6.glpk.glp'...
1091 lines were written
GLPK Integer Optimizer, v4.65
32 rows, 256 columns, 512 non-zeros
256 integer variables, all of which are binary
Preprocessing...
32 rows, 256 columns, 512 non-zeros
256 integer variables, all of